# 05 — Backtesting & Evaluation
Walk-forward simulation comparing LP-optimised, equal-weight, and buy-and-hold strategies.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from ml.training.backtest import run_backtest
from ml.training.evaluate import compare_models

sns.set_theme(style='darkgrid')
RESULTS_DIR = Path('../data/backtest_results')
print('Setup complete.')

## 1. Run Walk-Forward Backtest

In [ ]:
results = run_backtest(
    budget=10_000.0,
    risk_tolerance='medium',
    rebalance_every=1,    # daily rebalancing
)
print('Backtest complete.')

## 2. Performance Summary Table

In [ ]:
rows = []
for strategy, m in results['performance'].items():
    rows.append({'Strategy': strategy.replace('_', ' ').title(), **m})

perf_df = pd.DataFrame(rows).set_index('Strategy')
perf_df.columns = ['Return %', 'Sharpe', 'Max DD %', 'Ann. Vol %', 'Final $', 'Initial $']
perf_df.style.format({
    'Return %': '{:.2f}', 'Sharpe': '{:.3f}',
    'Max DD %': '{:.2f}', 'Ann. Vol %': '{:.2f}',
    'Final $': '${:,.2f}', 'Initial $': '${:,.2f}'
}).background_gradient(subset=['Return %', 'Sharpe'], cmap='RdYlGn')

## 3. Equity Curves

In [ ]:
curves = results['equity_curves']
colors = {'lp_optimised': '#0ea5e9', 'equal_weight': '#f59e0b', 'buy_and_hold': '#10b981'}

plt.figure(figsize=(14, 5))
for strategy, equity in curves.items():
    label = strategy.replace('_', ' ').title()
    plt.plot(equity, label=label, color=colors[strategy], linewidth=1.8)

plt.axhline(y=10_000, color='gray', linestyle=':', linewidth=1, label='Initial Budget')
plt.title('Portfolio Equity Curves — Walk-Forward Backtest', fontsize=13)
plt.xlabel('Test Step (Days)')
plt.ylabel('Portfolio Value (USD)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.legend()
plt.tight_layout()
plt.savefig('../data/backtest_results/equity_curves.png', dpi=150)
plt.show()

## 4. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (strategy, equity) in zip(axes, curves.items()):
    eq = np.array(equity)
    peak = np.maximum.accumulate(eq)
    dd   = (eq - peak) / (peak + 1e-8) * 100
    ax.fill_between(range(len(dd)), dd, 0, alpha=0.6, color=colors[strategy])
    ax.plot(dd, color=colors[strategy], linewidth=1)
    ax.set_title(strategy.replace('_', ' ').title())
    ax.set_xlabel('Test Step')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

axes[0].set_ylabel('Drawdown')
plt.suptitle('Drawdown Analysis by Strategy', fontsize=13)
plt.tight_layout()
plt.savefig('../data/backtest_results/drawdown.png', dpi=150)
plt.show()

## 5. LSTM Forecast Accuracy

In [ ]:
rows = []
for pair, m in results['forecast_metrics'].items():
    rows.append({
        'Pair': pair.replace('_', '/'),
        'RMSE': m['rmse'],
        'MAE':  m['mae'],
        'Directional Accuracy %': m['directional_accuracy'] * 100,
    })

forecast_df = pd.DataFrame(rows).set_index('Pair')
forecast_df.style.format({
    'RMSE': '{:.6f}', 'MAE': '{:.6f}', 'Directional Accuracy %': '{:.1f}'
}).background_gradient(subset=['Directional Accuracy %'], cmap='RdYlGn')

## 6. Rebalancing Frequency Sensitivity

In [ ]:
import logging
logging.disable(logging.CRITICAL)

freq_results = []
for freq, label in [(1, 'Daily'), (5, 'Weekly'), (21, 'Monthly')]:
    r = run_backtest(budget=10_000, risk_tolerance='medium', rebalance_every=freq)
    m = r['performance']['lp_optimised']
    freq_results.append({
        'Rebalance': label,
        'Return %':  m['total_return_pct'],
        'Sharpe':    m['sharpe_ratio'],
        'Max DD %':  m['max_drawdown_pct'],
        'Final $':   m['final_value'],
    })

logging.disable(logging.NOTSET)
pd.DataFrame(freq_results).set_index('Rebalance').style.format({
    'Return %': '{:.2f}', 'Sharpe': '{:.3f}', 'Max DD %': '{:.2f}', 'Final $': '${:,.2f}'
})

## 7. Risk Tolerance Sensitivity

In [ ]:
logging.disable(logging.CRITICAL)
risk_results = []
for tol in ['low', 'medium', 'high']:
    r = run_backtest(budget=10_000, risk_tolerance=tol, rebalance_every=1)
    m = r['performance']['lp_optimised']
    risk_results.append({'Risk': tol.title(), **m})

logging.disable(logging.NOTSET)
risk_df = pd.DataFrame(risk_results).set_index('Risk')
risk_df.columns = ['Return %', 'Sharpe', 'Max DD %', 'Ann. Vol %', 'Final $', 'Initial $']
risk_df.style.format({
    'Return %': '{:.2f}', 'Sharpe': '{:.3f}',
    'Max DD %': '{:.2f}', 'Ann. Vol %': '{:.2f}', 'Final $': '${:,.2f}'
}).background_gradient(subset=['Sharpe'], cmap='RdYlGn')

## 8. Export Summary Report

In [ ]:
summary_path = RESULTS_DIR / 'performance_summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        saved = json.load(f)
    print('Saved performance summary:')
    print(json.dumps(saved['performance'], indent=2))